In [11]:
from multiprocessing import resource_tracker

from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Literal, Sequence
from IPython.display import display
from langchain_deepseek import ChatDeepSeek
from dotenv import load_dotenv
from langchain.tools import tool
from loguru import logger
from langgraph.types import Send, Command
from langchain.messages import HumanMessage, AIMessage, SystemMessage
load_dotenv(override=True)

model = ChatDeepSeek(
    model="deepseek-v4-flash",
    # temperature=0.7,
    extra_body={
        "thinking": {
            "type": "disabled",
        }
    }
)
@tool(parse_docstring=True)
def get_weather(city: str = "上海"):
    """
    查询指定城市的天气

    Args:
        city: 城市名称
    """
    return f"{city}的天气是晴朗的。"

@tool(parse_docstring=True)
def get_news(domain:Literal["AI","食品安全"]):
    """
    查询新闻

    Args:
        domain: 查询新闻
    """
    if domain == "AI":
        return "AI is developed in very quick speed..."
    elif domain == "食品安全":
        return "The food security situation is more and more serious..."
    else:
        return "Unknow news area"


#绑定工具到大模型
tools = [get_weather, get_news]

model_with_tool = model.bind_tools(tools=tools)
res = model_with_tool.invoke("上海的天气怎么样？")
print(res)


content='我来帮你查询上海的天气情况。' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 51, 'prompt_tokens': 343, 'total_tokens': 394, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 343}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'a26a7955944dc5c60445bff77fac9c8e', 'id': '447e6320-efdd-4d08-9dde-3fa2bb99407e', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--01a0533f-5039-74b3-8c2a-dff886a6568a-0' tool_calls=[{'name': 'get_weather', 'args': {'city': '上海'}, 'id': 'call_00_Jqnl1ppGXFFja2J5J6dB3075', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 343, 'output_tokens': 51, 'total_tokens': 394, 'input_token_details': {'cache_read': 0}, 'output_token_details': {}}


In [16]:
from langchain import messages
from langchain_core.messages import ToolMessage
from random import randint
from langgraph.graph import MessagesState


#输入节点，将用户输入的查询信息，记录到message中，方便后续的大模型调用
class OverAllState(MessagesState):
    user_input: str
    final_output: str

def input_node(state: OverAllState):
    return {
        "message": [HumanMessage(state["user_input"])]
    }

def llm_node(state: OverAllState):
    ai_msg = model_with_tool.invoke(state["messages"])
    return {
        "message": ai_msg
    }

#判断是否需要调用tool
def tool_node(state: OverAllState) -> OverAllState:
    ai_msg = state["messages"][-1]
    tool_calls = ai_msg.tool_calls
    fail_prob = 6 # 模拟工具调用失败概率
    for tool_call in tool_calls:
        if tool_call["name"] == "get_weather":# 生成一个                                          0-9数字判断大小来模拟
            if randint(0,9) < fail_prob:
                messages.append(ToolMessage(
                    content="网络波动，调用失败，请重试",
                    tool_call_id = tool_call["id"]
                ))
            else:
                messages.append(get_weather.invoke(tool_call))
         if tool_call["name"] == "get_news":# 生成一个0-9数字判断大小来模拟
            if randint(0,9) < fail_prob:
                messages.append(ToolMessage(
                    content="网络波动，调用失败，请重试",
                    tool_call_id = tool_call["id"]
                ))
            else:
                messages.append(get_news.invoke(tool_call))






IndentationError: unindent does not match any outer indentation level (<string>, line 37)